# HW3: Campus Edge

**COMPSS 211A | Fall 2026 | Student copy**

In this homework you work with real 311 service requests from the streets around campus. You will send a request to the city's API, check two saved files against each other, and answer the Studio's questions with SQL. You will join your data to a second download made a month later and find out why the numbers changed. Last, you will use Gemini to sort the request types into categories and compare its labels with your own.

**Due:** Sunday, October 25, 2026 at 11:59 p.m.

Fill in every cell marked **Your turn** and answer the written questions. When a cell asks for a prediction, write your guess first.

Keep the variable names we ask for, because we look for them when grading.

## Scenario

311 is Berkeley's non-emergency service line. People use it to report things like illegal dumping, noise, or a tree that needs pruning. Each report becomes a *case* with its own ID number.

The fictional Berkeley Methods Studio wants a short note on what kinds of requests people make near campus. The data for Tasks 2 to 5 is already saved in `data/`. Task 1 sends one small request to the city's API, which needs no key. Task 5 needs your Gemini API key.

## What you will practice

- Sending an API request, checking its status, and turning the response into a table (Week 6).
- Checking that two files which should match actually do.
- Writing SQL queries that filter, count, and sort (Week 5).
- Joining two tables on a shared column in SQL (Week 5).
- Labelling text with Gemini using a prompt template and a schema, and checking the labels by hand (Week 7).
- Saying what a dataset can and cannot tell you.

## Helpful references

- **Lab 5** (`lab/lab05_sql_from_python.ipynb`) has every kind of query you need for Tasks 3 to 5: `SELECT`, `WHERE`, `GROUP BY`, `ORDER BY`, `LIMIT`, `IS NULL`, and `LEFT JOIN`. The `sql("""...""")` helper in this notebook works the same way.
- **SQL drills** in the [practice studio](https://macss-berkeley.github.io/compss-211a/practice/) check your answer as you go.
- **Lab 6** (`lab/lab06_openalex_api_acquisition.ipynb`) and the [Week 6 API workspace](https://macss-berkeley.github.io/compss-211a/interactives/week06-api-workspace.html) cover requests, status codes, and paging.
- **Week 7 live workspace** (`lessons/week07_language-model-apis/week07_live_workspace.ipynb`) has the prompt template, the schema, and the labelling loop you need for Task 5.
- **IDs are text, not numbers.** Load `case_id` as text (`dtype={"case_id": str}`).

## What you need

- **Your own Gemini API key in your `.env` file** for Task 5 (instructions in `SETUP.md`, under "API keys"). In Colab, use a Colab Secret named `GEMINI_API_KEY`. Never paste the key into the notebook.
- Tasks 1 to 4 do not need the key. You can start them before the Week 7 class.
- Task 5 makes 15 short calls. The results are saved to a file after the first run, so you only need to call Gemini once.

## AI and collaboration policy

You may use AI to help you understand an error message, as long as you say so in a note in your notebook. Write the code and the written answers yourself. Task 5 asks you to call Gemini from your code; that is part of the assignment. Talking through ideas with classmates is fine; share ideas, not code.

In [ ]:
from pathlib import Path
import json
import os
import sqlite3
import sys
import time

import pandas as pd
import requests
from IPython.display import display

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## 1. Send an API request

The data comes from the City of Berkeley's API. A request is a web address plus a set of *parameters* that tell the server which records you want. In Python we store the parameters in a dictionary.

The cell below shows the request we used for the saved files. This API sends back at most 1,000 records per request. Our data has 1,896 records, so we had to ask twice. The `$offset` parameter says how many records to skip: the first request uses 0, and the second uses 1000. This is called *paging*.

The parameter names `$select`, `$where`, `$order`, and `$limit` work like the SQL you wrote in Week 5. The server runs your request against the city's database and sends back the matching rows.

In [ ]:
API_ENDPOINT = "https://data.cityofberkeley.info/resource/p88g-6gs2.json"

query_params = {
    # Which columns to send back.
    "$select": "case_id, date_opened, case_status, last_action_date, date_closed, "
               "case_request, city, state, latitude, longitude",
    # Which rows to keep: cases opened in 2025, inside a box around campus.
    "$where": (
        "date_opened >= '2025-01-01' AND date_opened < '2026-01-01' "
        "AND latitude BETWEEN 37.862 AND 37.884 "
        "AND longitude BETWEEN -122.274 AND -122.244"
    ),
    # Sort by case ID so that every page comes back in the same order.
    "$order": "case_id",
    # At most 1,000 records per request.
    "$limit": 1000,
    # How many records to skip before starting.
    "$offset": 0,
}

query_params

In [ ]:
# Your turn: send a small request.
# 1. Make a copy of query_params called small_params. Set "$limit" to 5 in the copy.
# 2. Send it with requests.get(API_ENDPOINT, params=small_params, timeout=30). Save the result as response.
# 3. Print response.status_code. 200 means the request worked.
# 4. Save response.json() as records. Print how many records came back, and display the first one.


**If the request fails** (no connection, or a status other than 200), run the line below instead and say so in your response. It reads the first five records from the saved file.

```python
records = json.loads(course_data_path("hw3_campus_edge_311_2025.json").read_text())[:5]
```

In [ ]:
# Your turn: make a table from the records.
# 1. Loop over records. For each one, add a dictionary to rows with case_id, date_opened, and case_request.
# 2. Make a DataFrame called live_table from rows and display it.
rows = []


In [ ]:
# Your turn: make the parameters for the SECOND page of the full request.
# 1. Make a copy of query_params called second_page_params.
# 2. Change "$offset" in the copy so it skips the first 1,000 records.
# 3. Display second_page_params. You do not need to send it.


### Your response


Answer in two or three sentences, in your own words:

1. Which records does `$where` keep, and which does it leave out?
2. If `$limit` were 500 instead of 1000, how many requests would it take to get all 1,896 records, and what `$offset` would each one use?
3. You sent your request today. The saved files were downloaded on August 23, 2026. Give one reason your five records could differ from the first five in the saved file.


> Write your response here, then delete this line.

## 2. Load the saved files

The folder `data/` has the August 23 download saved two ways. CSV is what you get when someone exports a table. JSON is what the API sends back. The cell below loads both.

Two files that should hold the same records don't always do. A blank value can be saved differently in each format. Check before you analyze anything.

In [ ]:
# Supplied: load the same records from both files.
csv_rows = pd.read_csv(course_data_path("hw3_campus_edge_311_2025.csv"), dtype={"case_id": str})
json_rows = pd.read_json(course_data_path("hw3_campus_edge_311_2025.json"), dtype={"case_id": str})
print("csv", csv_rows.shape)
print("json", json_rows.shape)

In [ ]:
# Your turn: check that the two files match.
# 1. Make same_ids: True if the set of case_id values is the same in both tables.
# 2. Make csv_missing and json_missing: the number of date_closed values pandas counts as missing
#    in each table (.isna().sum()). Print all three.


# Your turn: the two missing counts disagree. A case that is not closed yet has no closing date.
# 3. Display the first few date_closed values of each table to see how each format writes "no date".
# 4. Fix json_rows so pandas counts its blank dates as missing. .replace("", pd.NA) turns empty
#    strings into pandas' missing value. Print the missing count again. It should match the CSV.


## 3. Count the request types with SQL

Which kinds of requests come up most often near campus?

The cell below puts the August records into a small database, the same way Lab 5 did. The table is called `august`. `sql("""...""")` runs a query and gives the answer back as a pandas table.

In [ ]:
# Supplied: put the August records in a database table called "august".
if "database" in globals():
    database.close()
database = sqlite3.connect(":memory:")
csv_rows.to_sql("august", database, index=False)

def sql(query):
    return pd.read_sql_query(query, database)

sql("SELECT * FROM august LIMIT 3")

**Your turn:**

1. Write a query for the **8 most common** request types. It should return two columns: `case_request` and `requests` (the number of cases of that type). Put the largest count first. When two counts tie, sort those rows by `case_request` from A to Z. Save the result as `request_summary`.
2. Add a `share` column to `request_summary` in pandas: `requests` divided by the number of rows in the **whole** August table (not just the top 8).
3. Now look only at the cases that are **still open** (`case_status` is `'Open'`). Copy your query, add one line, and keep the top 5. Save the result as `open_summary`. Add a `share` column here too. This time divide by the number of open cases, and get that number with a query of its own.

Then run the check cell. It counts the same things in pandas and compares.

In [ ]:
# Your turn: request_summary (the query, then the share column)
request_summary = ...


# Your turn: open_summary (the query, the number of open cases, then the share column)
open_summary = ...

In [ ]:
# Check: pandas should give the same counts as your SQL.
pandas_counts = csv_rows["case_request"].value_counts()
pandas_open_counts = csv_rows.loc[csv_rows["case_status"] == "Open", "case_request"].value_counts()

assert list(request_summary.columns) == ["case_request", "requests", "share"]
assert request_summary["requests"].tolist() == pandas_counts.head(8).tolist()
assert open_summary["requests"].tolist() == pandas_open_counts.head(5).tolist()
print("SQL and pandas agree.")

### Your response


Answer in two or three sentences: in `open_summary`, what number is `share` divided by? Give one reason why "share of open cases" could tell a different story from "share of all cases".


> Write your response here, then delete this line.

## 4. One month later: join two extracts

Your snapshot was downloaded on **August 23, 2026**. On **September 23, 2026** we ran the exact same request again and saved the result as `hw3_campus_edge_311_2025_refresh.csv`. Same query, same year, same area.

To find out what changed, you'll **join** the two tables on `case_id`, as in Week 5. There are three possibilities for each case:

- it is in **both** extracts;
- it is only in the **August** snapshot (it disappeared);
- it is only in the **September** refresh (it is new).

Remember what `LEFT JOIN` does: it keeps every row of the first table. When a row has no match in the second table, the second table's columns come back empty (`NULL`). So you can find the rows with no match by asking where the second table's `case_id` `IS NULL`.

**Predict first:** how many rows do you think the September refresh has? Write your guess in the next cell, then run it.

In [ ]:
# My prediction for the number of rows in the September refresh:
#

# Supplied: load the September refresh and add it to the database as "september".
refresh_rows = pd.read_csv(course_data_path("hw3_campus_edge_311_2025_refresh.csv"), dtype={"case_id": str})
refresh_rows.to_sql("september", database, index=False, if_exists="replace")
print(len(csv_rows), "cases in the August table")
print(len(refresh_rows), "cases in the September table")

In [ ]:
# Worked example: cases that are in September but were NOT in August.
# Start FROM september, so every September case is kept.
# Where August has no matching case, a.case_id is NULL.
new_cases = sql("""
    SELECT s.case_id, s.date_opened, s.case_status, s.last_action_date
    FROM september AS s
    LEFT JOIN august AS a
        ON s.case_id = a.case_id
    WHERE a.case_id IS NULL
""")
print(len(new_cases), "new case(s)")
new_cases

**Your turn:**

1. Flip the worked example. Make `disappeared`: the `case_id`, `case_status`, and `last_action_date` of every August case that is **not** in September. How many are there?
2. Find the rule that decides which cases disappeared. Write one query for the **latest** `last_action_date` among the cases that disappeared, and another for the **earliest** `last_action_date` in the whole September table. Hints: `MAX(column)` and `MIN(column)` work like `COUNT` and `AVG`. The dates are stored as text such as `2025-09-18T00:00:00.000`. Text written year-month-day sorts in date order, so `MIN` and `MAX` give the right answer.

Then run the check cell. It makes the same comparison in pandas.

In [ ]:
# Your turn 1: disappeared
disappeared = ...


# Your turn 2: the two dates


In [ ]:
# Check: the same comparison in pandas. how="outer" keeps all three kinds of rows,
# and indicator=True adds a _merge column that says which kind each row is.
snapshot_comparison = csv_rows.merge(
    refresh_rows, on="case_id", how="outer",
    suffixes=("_aug", "_sep"), indicator=True, validate="one_to_one",
)
merge_counts = snapshot_comparison["_merge"].value_counts()
print(merge_counts)

assert len(disappeared) == merge_counts["left_only"]
assert len(new_cases) == merge_counts["right_only"]
assert len(csv_rows) == merge_counts["both"] + len(disappeared)
assert len(refresh_rows) == merge_counts["both"] + len(new_cases)
print("Your SQL matches pandas, and every case is accounted for.")

### Your response


Answer in three or four sentences:

1. What rule does the city's portal seem to use to remove cases? What evidence points to it?
2. Your `request_summary` in Task 3 came from the August snapshot. If a colleague ran the same code today, would they get the same numbers? What should every report built on live open data write down so it can be checked later?


> Write your response here, then delete this line.

## 5. Group the request types with Gemini

The data has 78 request types, and several are close to each other (`Illegal Dumping - City Property`, `Mattress Program`). The Studio wants a shorter list. You will sort the 15 most common types into six categories twice: once yourself, and once with Gemini. Then you compare the two.

Only the names of the request types are sent to Gemini, not the cases.

In [ ]:
# Supplied: the 15 most common request types, and the six categories with their definitions.
top_types = sql("""
    SELECT case_request, COUNT(*) AS requests
    FROM august
    GROUP BY case_request
    ORDER BY requests DESC, case_request
    LIMIT 15
""")

CATEGORIES = {
    "waste_and_cleaning": "dumped items, trash and recycling pickup, and cleaning of public places",
    "streets_and_vehicles": "road surfaces, street lights, traffic, parking, and vehicles",
    "trees": "street and park trees",
    "water_and_sewer": "storm drains, sewers, and flooding",
    "noise": "noise complaints",
    "general": "general questions, or requests that do not name a specific problem",
}
top_types

**Your turn:** label the 15 types yourself before you call Gemini. Put one category name from `CATEGORIES` after each type.

In [ ]:
# Your turn
my_category = {
    "Illegal Dumping - City Property": "",
    "Miscellaneous": "",
    "Noise Complaint": "",
    "Abandoned Vehicle - Ready for PD (Parking Enforcement)": "",
    "Direct Line General - 311": "",
    "Mattress Program": "",
    "Tree Pruning": "",
    "Miscellaneous - Transportation": "",
    "Missed Trash Pickup - Commercial": "",
    "Potholes": "",
    "Storm Drain Problem/Maintenance": "",
    "Sewer Maintenance": "",
    "Street Lights - Repair": "",
    "Broken Branch or Tree": "",
    "Pressure Washing": "",
}

assert set(my_category) == set(top_types["case_request"])
assert set(my_category.values()) <= set(CATEGORIES), "Use the category names in CATEGORIES."

In [ ]:
GEMINI_MODEL = "gemini-3.6-flash"  # use the replacement model if bCourses announces one


def load_api_key(name):
    """Look for an API key in .env, then your environment, then Colab Secrets.

    Returns None if there isn't one. Never prints the key.
    """
    try:
        from dotenv import find_dotenv, load_dotenv
        env_file = find_dotenv(usecwd=True)  # searches this folder, then the folders above it
        if env_file:
            load_dotenv(env_file)
    except ImportError:
        pass
    key = os.getenv(name)
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get(name)
        except Exception:
            key = None
    if key and key.startswith("paste-"):
        key = None  # the placeholder from .env.example, not a real key
    return key


print("Gemini key found:", load_api_key("GEMINI_API_KEY") is not None)

# The system instruction holds the task and the category definitions. It is the same for every request type.
SYSTEM = (
    "You sort City of Berkeley 311 request types into categories.\n\nCategories:\n"
    + "\n".join(f"- {name}: {definition}" for name, definition in CATEGORIES.items())
    + "\n\nGive the request type one category and a reason of one short sentence."
)
print(SYSTEM)

# The schema lists the fields Gemini must return and the category names it may use.
LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": list(CATEGORIES)},
        "reason": {"type": "string"},
    },
    "required": ["category", "reason"],
}


def ask_gemini(client, prompt):
    """Send one prompt with SYSTEM and LABEL_SCHEMA. Return the reply text, which is JSON."""
    from google.genai import types

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM,
                    temperature=0,
                    response_mime_type="application/json",
                    response_schema=LABEL_SCHEMA,
                ),
            )
            return response.text
        except Exception as error:
            print(f"Attempt {attempt + 1} failed: {type(error).__name__}. Retrying...")
            time.sleep(5 * (attempt + 1))
    raise RuntimeError("Gemini did not answer after three tries.")

**Your turn:**

1. Write `TEMPLATE`, a string with a `{request_type}` slot. The system instruction already holds the task, so the template only needs to present the request type.
2. Finish `label_types(request_types, client)`. For each request type, it should fill in `TEMPLATE`, call `ask_gemini`, read the reply with `json.loads`, and add one dictionary to `rows` with `case_request`, `category`, and `reason`.

The cell after it runs your function once and saves the results to `generated/hw3_type_labels.csv`. If that file already exists, it loads it instead of calling Gemini again. Delete the file if you want to run the calls again.

In [ ]:
# Your turn
TEMPLATE = ...


def label_types(request_types, client):
    """Ask Gemini about every request type. Return one row per type."""
    rows = []
    for request_type in request_types:
        # Your turn: build the prompt, ask Gemini, read the JSON, and append a dictionary to rows.
        ...
        print(len(rows), "of", len(request_types), "done")
    return pd.DataFrame(rows)

In [ ]:
LABELS_PATH = GENERATED_DIR / "hw3_type_labels.csv"

if LABELS_PATH.exists():
    type_labels = pd.read_csv(LABELS_PATH)
    print("Loaded saved results from", LABELS_PATH.name)
else:
    from google import genai
    key = load_api_key("GEMINI_API_KEY")
    if not key:
        raise RuntimeError("No Gemini key found. Add it to your .env file (see SETUP.md, 'API keys').")
    client = genai.Client(api_key=key)
    type_labels = label_types(top_types["case_request"].tolist(), client)
    type_labels["model"] = GEMINI_MODEL
    type_labels.to_csv(LABELS_PATH, index=False)
    print("Saved results to", LABELS_PATH.name)

type_labels

**Your turn:**

1. Add a column `my_category` to `type_labels` with your own labels (hint: `type_labels["case_request"].map(my_category)`). Add a column `agree` that is `True` where your label and Gemini's are the same. Print how many types you disagree on.
2. Put `type_labels` into the database as a table called `categories` (hint: `type_labels.to_sql("categories", database, index=False, if_exists="replace")`).
3. Write a query that joins `august` to `categories` on `case_request` with a `LEFT JOIN`, and counts the cases in each category, largest first. Return two columns, `category` and `cases`. Save the result as `category_summary`.

A case whose request type is not among the 15 has no match in `categories`. The `LEFT JOIN` keeps it, with an empty category. Those cases form one row of their own in `category_summary`.

In [ ]:
# Your turn 1: compare your labels with Gemini's


# Your turn 2: add the categories table to the database


# Your turn 3: category_summary
category_summary = ...


In [ ]:
# Check: every August case is counted once, and every category name is allowed.
assert list(category_summary.columns) == ["category", "cases"]
assert category_summary["cases"].sum() == len(csv_rows)
assert set(type_labels["category"]) <= set(CATEGORIES)
assert {"my_category", "agree"} <= set(type_labels.columns)
print("Every case is accounted for.")

### Your response


Answer in three or four sentences:

1. On which request types did you and Gemini disagree? Pick one and say which label you would keep, and why. If you agreed on all 15, name the type you found hardest to place.
2. How many cases have no category, and what does that mean for anyone reading `category_summary`?
3. A classmate may get different labels from Gemini for the same types. What did this notebook save so that your counts can be checked later?


> Write your response here, then delete this line.

## 6. Write a short methods note

Write five to seven sentences for the Berkeley Methods Studio. Include:

1. **Do the files match?** Say what you checked in Task 2 and what differed between the formats.
2. **What the summary shows.** Name the most common request type and its share.
3. **One month later.** What did the join show, and what does it mean for anyone who reuses your numbers?
4. **The categories.** How were they made, and how did you check them?
5. **One limit.** What can 311 data *not* tell you? For example: a report shows that someone chose to call the city, not how often a problem actually happens.

### Your response

**Question:** What evidence shows the files agree, what changed a month later, how far can the categories be trusted, and what are the limits of this data?

> Write your response here, then delete this line.